# Progressive Depth Growing — Int6 SOTA Experiment

**Concept**: Start training with 7 layers (faster ~55ms steps) for the first 33% of wallclock, then switch to full 11 layers. This gives ~16% more total data exposure within the same 600s budget.

**Experiments**:
1. `baseline` — unmodified SOTA (no growing)
2. `grow_33pct_7L` — grow from 7→11 layers at 33% wallclock
3. `grow_25pct_7L` — grow at 25% wallclock (less shallow time)
4. `grow_33pct_5L` — grow from 5→11 layers (more aggressive)

Each run uses the real SOTA script patched with progressive growing support.

## Cell 1: Setup — Clone repos, install deps, download data

In [ ]:
import os, shutil, glob, time, json, subprocess

# Mount Drive for persistent logs
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/parameter-golf'
os.makedirs(f'{DRIVE_DIR}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)

OFFICIAL = '/content/parameter-golf'
AUX = '/content/parameter-golf-aux'

%cd /content

# Clone repos if not present
if not os.path.exists(OFFICIAL):
    !git clone --depth 1 https://github.com/openai/parameter-golf.git
if not os.path.exists(AUX):
    !git clone https://github.com/jamesconde/parameter-golf-aux.git

# Pull latest
%cd {AUX}
!git pull

# Install deps
!pip install -q sentencepiece huggingface-hub datasets tqdm zstandard 2>/dev/null

# Try to install Flash Attention 3 (Hopper only), fall back to FA2
import torch
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({vram_gb:.0f}GB)")

# Flash attention - try FA3 first (H100), then FA2
try:
    from flash_attn_interface import flash_attn_func
    print("Flash Attention 3: OK (Hopper)")
except ImportError:
    print("FA3 not available, installing FA2...")
    !pip install -q flash-attn --no-build-isolation 2>/dev/null
    try:
        from flash_attn import flash_attn_func
        print("Flash Attention 2: OK")
    except ImportError:
        print("WARNING: No Flash Attention available, will use SDPA fallback (slower)")

# Download data if not present
data_dir = f'{OFFICIAL}/data/datasets/fineweb10B_sp1024'
if not os.path.exists(f'{data_dir}/fineweb_val_000000.bin'):
    %cd {OFFICIAL}
    !python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1
    print("Data downloaded")
else:
    n_shards = len(glob.glob(f'{data_dir}/fineweb_train_*.bin'))
    print(f"Data OK: {n_shards} training shard(s)")

print("\nSetup complete.")

## Cell 2: Patch the SOTA script

Copies the SOTA `train_gpt.py` into the official repo, then applies the progressive growing patch.

In [ ]:
import sys, shutil
sys.path.insert(0, AUX)

SOTA_DIR = f'{OFFICIAL}/records/track_10min_16mb/2026-03-25_ValCalib_GPTQ_XSA_BigramHash3072'
SOTA_SCRIPT = f'{SOTA_DIR}/train_gpt.py'

# Copy SOTA script to working location
WORK_SCRIPT = f'{OFFICIAL}/train_gpt_progressive.py'
shutil.copy2(SOTA_SCRIPT, WORK_SCRIPT)
print(f"Copied SOTA script ({os.path.getsize(SOTA_SCRIPT)} bytes)")

# Apply progressive growing patch
from progressive_growing.patch_progressive import patch

with open(WORK_SCRIPT) as f:
    source = f.read()

patched = patch(source)

with open(WORK_SCRIPT, 'w') as f:
    f.write(patched)

# Verify all patches applied
checks = {
    'FA fallback': '_fa_dispatch' in patched,
    'SDP backends': 'enable_mem_efficient_sdp(True)' in patched,
    'USE_COMPILE': 'USE_COMPILE' in patched,
    'GROW_FRACTION': 'GROW_FRACTION' in patched,
    'GROW_INITIAL_LAYERS': 'GROW_INITIAL_LAYERS' in patched,
    'run_shallow_step': 'run_shallow_step' in patched,
    'Growing state': '_growing = args.grow_fraction' in patched,
    'Bank restoration': 'restored dormant banks' in patched,
}

all_ok = True
for name, ok in checks.items():
    status = "OK" if ok else "FAILED"
    if not ok:
        all_ok = False
    print(f"  {status}: {name}")

# Syntax check
import py_compile
try:
    py_compile.compile(WORK_SCRIPT, doraise=True)
    print(f"\nSyntax check: OK")
except py_compile.PyCompileError as e:
    print(f"\nSyntax ERROR: {e}")
    all_ok = False

orig_lines = source.count('\n')
patched_lines = patched.count('\n')
print(f"Lines: {orig_lines} -> {patched_lines} (+{patched_lines - orig_lines})")

if all_ok:
    print("\nAll patches applied successfully. Ready to run.")
else:
    print("\nWARNING: Some patches failed! Check output above.")

## Cell 3: Config

GPU-aware batch size selection and experiment definitions.

In [ ]:
import torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

# Batch tokens: match SOTA settings per GPU tier
if vram_gb >= 70:
    BATCH_TOKENS = 786432
    print(f"A100-80GB: batch_tokens={BATCH_TOKENS}")
elif vram_gb >= 35:
    BATCH_TOKENS = 393216
    print(f"A100-40GB: batch_tokens={BATCH_TOKENS}")
else:
    BATCH_TOKENS = 196608
    print(f"Small GPU ({vram_gb:.0f}GB): batch_tokens={BATCH_TOKENS}")

# Common env vars matching the SOTA submission
BASE_ENV = {
    'BIGRAM_VOCAB_SIZE': '3072',
    'BIGRAM_DIM': '112',
    'WARMDOWN_ITERS': '4000',
    'TARGET_MB': '15.9',
    'TRAIN_BATCH_TOKENS': str(BATCH_TOKENS),
    'VAL_LOSS_EVERY': '500',
    'TRAIN_LOG_EVERY': '50',
    'MAX_WALLCLOCK_SECONDS': '600',
    'USE_COMPILE': '1',
}

# Experiment definitions
EXPERIMENTS = [
    {
        'name': 'baseline',
        'description': 'Unmodified SOTA (no growing)',
        'env': {'GROW_FRACTION': '0'},
    },
    {
        'name': 'grow_33pct_7L',
        'description': '7->11 layers at 33% wallclock (200s)',
        'env': {'GROW_FRACTION': '0.33', 'GROW_INITIAL_LAYERS': '7'},
    },
    {
        'name': 'grow_25pct_7L',
        'description': '7->11 layers at 25% wallclock (150s)',
        'env': {'GROW_FRACTION': '0.25', 'GROW_INITIAL_LAYERS': '7'},
    },
    {
        'name': 'grow_33pct_5L',
        'description': '5->11 layers at 33% wallclock (aggressive)',
        'env': {'GROW_FRACTION': '0.33', 'GROW_INITIAL_LAYERS': '5'},
    },
]

SEEDS = [42, 314, 999]  # Match SOTA submission seeds

print(f"\n{len(EXPERIMENTS)} experiments x {len(SEEDS)} seeds = {len(EXPERIMENTS) * len(SEEDS)} total runs")
for exp in EXPERIMENTS:
    print(f"  {exp['name']}: {exp['description']}")

## Cell 4: Run experiment helper

Runs a single experiment with a given seed. Parses val_bpb from log output.
Single-GPU training (no torchrun) — the script handles single-GPU mode.

In [ ]:
import subprocess, re, time, os, json

def run_experiment(exp_name, exp_env, seed, script_path, base_env, log_dir):
    """Run a single training experiment and return parsed results."""
    env = os.environ.copy()
    env.update(base_env)
    env.update(exp_env)
    env['SEED'] = str(seed)
    env['RUN_ID'] = f'{exp_name}_s{seed}'

    log_file = f'{log_dir}/{exp_name}_s{seed}.txt'
    print(f"\n{'='*60}")
    print(f"Running: {exp_name} (seed={seed})")
    print(f"  Env: {exp_env}")
    print(f"  Log: {log_file}")
    print(f"{'='*60}")

    t0 = time.time()

    # Run training
    proc = subprocess.run(
        ['python3', script_path],
        env=env,
        cwd=os.path.dirname(script_path),
        capture_output=True,
        text=True,
        timeout=900,  # 15 min timeout (10 min training + overhead)
    )

    elapsed = time.time() - t0
    output = proc.stdout + '\n' + proc.stderr

    # Save full log
    with open(log_file, 'w') as f:
        f.write(output)

    # Parse results
    result = {
        'name': exp_name,
        'seed': seed,
        'elapsed_s': round(elapsed, 1),
        'returncode': proc.returncode,
    }

    # Find val_bpb values
    val_bpb_pattern = re.compile(r'val_bpb:(\d+\.\d+)')
    val_bpbs = [float(m.group(1)) for m in val_bpb_pattern.finditer(output)]
    if val_bpbs:
        result['final_val_bpb'] = val_bpbs[-1]
        result['best_val_bpb'] = min(val_bpbs)
        result['val_bpb_history'] = val_bpbs

    # Find step count
    step_pattern = re.compile(r'step:(\d+)/')
    steps = [int(m.group(1)) for m in step_pattern.finditer(output)]
    if steps:
        result['final_step'] = max(steps)

    # Find ms/step
    ms_pattern = re.compile(r'(\d+\.\d+)ms/step')
    ms_steps = [float(m.group(1)) for m in ms_pattern.finditer(output)]
    if ms_steps:
        result['avg_ms_step'] = round(sum(ms_steps) / len(ms_steps), 1)

    # Find growth transition
    grow_pattern = re.compile(r'GROW: (\d+)L -> (\d+)L at step (\d+)')
    grow_match = grow_pattern.search(output)
    if grow_match:
        result['grow_step'] = int(grow_match.group(3))
        result['grow_from'] = int(grow_match.group(1))
        result['grow_to'] = int(grow_match.group(2))

    # Print summary
    if proc.returncode != 0:
        print(f"  FAILED (returncode={proc.returncode})")
        # Print last 20 lines of output for debugging
        lines = output.strip().split('\n')
        for line in lines[-20:]:
            print(f"  | {line}")
    else:
        bpb = result.get('final_val_bpb', '?')
        steps = result.get('final_step', '?')
        ms = result.get('avg_ms_step', '?')
        print(f"  val_bpb={bpb}  steps={steps}  ms/step={ms}  time={elapsed:.0f}s")
        if 'grow_step' in result:
            print(f"  Growth: {result['grow_from']}L -> {result['grow_to']}L at step {result['grow_step']}")

    return result

print("run_experiment() defined.")

## Cell 5: Quick smoke test (1 seed, short run)

Runs baseline + grow_33pct_7L with a 120s wallclock to verify everything works before committing to full runs.

In [ ]:
# Quick smoke test: 120s wallclock, 1 seed
SMOKE_DIR = f'{DRIVE_DIR}/logs/progressive_smoke'
os.makedirs(SMOKE_DIR, exist_ok=True)

smoke_env = dict(BASE_ENV)
smoke_env['MAX_WALLCLOCK_SECONDS'] = '120'
smoke_env['VAL_LOSS_EVERY'] = '200'
smoke_env['WARMDOWN_ITERS'] = '800'

smoke_results = []

# Baseline (no growing)
r = run_experiment('smoke_baseline', {'GROW_FRACTION': '0'},
                   seed=42, script_path=WORK_SCRIPT,
                   base_env=smoke_env, log_dir=SMOKE_DIR)
smoke_results.append(r)

# Progressive growing 33% 7L
r = run_experiment('smoke_grow_33pct', {'GROW_FRACTION': '0.33', 'GROW_INITIAL_LAYERS': '7'},
                   seed=42, script_path=WORK_SCRIPT,
                   base_env=smoke_env, log_dir=SMOKE_DIR)
smoke_results.append(r)

# Summary
print(f"\n{'='*60}")
print("SMOKE TEST SUMMARY")
print(f"{'='*60}")
for r in smoke_results:
    status = "OK" if r['returncode'] == 0 else "FAILED"
    bpb = r.get('final_val_bpb', 'N/A')
    steps = r.get('final_step', 'N/A')
    grow = f"  (grew at step {r['grow_step']})" if 'grow_step' in r else ""
    print(f"  {status}: {r['name']}  val_bpb={bpb}  steps={steps}{grow}")

if all(r['returncode'] == 0 for r in smoke_results):
    print("\nSmoke test passed! Ready for full experiment runs.")
else:
    print("\nSmoke test FAILED. Check logs above for errors.")

## Cell 6: Full experiment sweep (all experiments x all seeds)

Runs all 4 experiments x 3 seeds = 12 runs. Each run is 600s (10 min) + overhead.
Total time: ~2.5 hours on single GPU.

**Run this only after smoke test passes.**

In [ ]:
# Full experiment sweep
FULL_DIR = f'{DRIVE_DIR}/logs/progressive_full'
os.makedirs(FULL_DIR, exist_ok=True)

all_results = []
total_runs = len(EXPERIMENTS) * len(SEEDS)
run_idx = 0

for exp in EXPERIMENTS:
    for seed in SEEDS:
        run_idx += 1
        print(f"\n[{run_idx}/{total_runs}] ", end="")

        # Check if already completed (skip if log exists with val_bpb)
        log_file = f'{FULL_DIR}/{exp["name"]}_s{seed}.txt'
        if os.path.exists(log_file):
            with open(log_file) as f:
                content = f.read()
            if re.search(r'val_bpb:\d+\.\d+', content):
                # Parse existing result
                val_bpbs = [float(m.group(1)) for m in re.finditer(r'val_bpb:(\d+\.\d+)', content)]
                print(f"SKIP (already done): {exp['name']} s{seed}  val_bpb={val_bpbs[-1]}")
                all_results.append({
                    'name': exp['name'],
                    'seed': seed,
                    'final_val_bpb': val_bpbs[-1],
                    'best_val_bpb': min(val_bpbs),
                    'returncode': 0,
                    'skipped': True,
                })
                continue

        r = run_experiment(
            exp['name'], exp['env'],
            seed=seed,
            script_path=WORK_SCRIPT,
            base_env=BASE_ENV,
            log_dir=FULL_DIR,
        )
        all_results.append(r)

        # Save intermediate results to Drive
        with open(f'{DRIVE_DIR}/results/progressive_results.json', 'w') as f:
            # Remove val_bpb_history for compact JSON
            compact = [{k: v for k, v in r.items() if k != 'val_bpb_history'} for r in all_results]
            json.dump(compact, f, indent=2)

print(f"\n\nAll {total_runs} runs complete.")

## Cell 7: Results analysis

Statistical comparison of all experiments using Welch's t-test.

In [ ]:
import numpy as np
from scipy import stats

# Load results (from memory or from Drive)
try:
    results = all_results
except NameError:
    with open(f'{DRIVE_DIR}/results/progressive_results.json') as f:
        results = json.load(f)

# Group by experiment name
from collections import defaultdict
grouped = defaultdict(list)
for r in results:
    if r.get('returncode', 1) == 0 and 'final_val_bpb' in r:
        grouped[r['name']].append(r['final_val_bpb'])

print("Progressive Growing Results")
print("=" * 70)
print(f"{'Experiment':<25} {'Mean BPB':>10} {'Std':>8} {'Seeds':>6} {'Steps':>8}")
print("-" * 70)

baseline_bpbs = grouped.get('baseline', [])
baseline_mean = np.mean(baseline_bpbs) if baseline_bpbs else None

for name in ['baseline', 'grow_33pct_7L', 'grow_25pct_7L', 'grow_33pct_5L']:
    bpbs = grouped.get(name, [])
    if not bpbs:
        print(f"  {name:<25} {'N/A':>10}")
        continue

    mean = np.mean(bpbs)
    std = np.std(bpbs, ddof=1) if len(bpbs) > 1 else 0

    # Get avg steps from results
    exp_results = [r for r in results if r['name'] == name and r.get('returncode') == 0]
    avg_steps = np.mean([r.get('final_step', 0) for r in exp_results])

    delta = ""
    if baseline_mean is not None and name != 'baseline':
        diff = mean - baseline_mean
        sign = "+" if diff > 0 else ""
        delta = f"  ({sign}{diff:.4f})"

        # Welch's t-test vs baseline
        if len(bpbs) >= 2 and len(baseline_bpbs) >= 2:
            t_stat, p_val = stats.ttest_ind(bpbs, baseline_bpbs, equal_var=False)
            sig = "*" if p_val < 0.05 else ""
            delta += f"  p={p_val:.3f}{sig}"

    print(f"  {name:<25} {mean:>10.4f} {std:>8.4f} {len(bpbs):>6} {avg_steps:>8.0f}{delta}")

print("-" * 70)

# Print growth transition details
print("\nGrowth Transitions:")
for r in results:
    if 'grow_step' in r:
        print(f"  {r['name']} s{r['seed']}: {r['grow_from']}L -> {r['grow_to']}L at step {r['grow_step']}")

# Per-seed comparison
print("\nPer-Seed BPB:")
print(f"  {'Seed':>6}", end="")
for name in ['baseline', 'grow_33pct_7L', 'grow_25pct_7L', 'grow_33pct_5L']:
    print(f"  {name:>16}", end="")
print()
for seed in SEEDS:
    print(f"  {seed:>6}", end="")
    for name in ['baseline', 'grow_33pct_7L', 'grow_25pct_7L', 'grow_33pct_5L']:
        r = [x for x in results if x['name'] == name and x.get('seed') == seed]
        if r and 'final_val_bpb' in r[0]:
            print(f"  {r[0]['final_val_bpb']:>16.4f}", end="")
        else:
            print(f"  {'N/A':>16}", end="")
    print()

## Cell 8: Save results to Drive

In [ ]:
# Save final results
with open(f'{DRIVE_DIR}/results/progressive_results.json', 'w') as f:
    compact = [{k: v for k, v in r.items() if k != 'val_bpb_history'} for r in all_results]
    json.dump(compact, f, indent=2)
print(f"Results saved to {DRIVE_DIR}/results/progressive_results.json")

# Generate markdown report
report = ["# Progressive Depth Growing — Results\n"]
report.append(f"Date: {time.strftime('%Y-%m-%d %H:%M')}\n")
report.append(f"GPU: {torch.cuda.get_device_name(0)}\n")
report.append(f"Batch tokens: {BATCH_TOKENS}\n\n")

report.append("| Experiment | Seed | val_bpb | Steps | ms/step | Growth Step |\n")
report.append("|---|---|---|---|---|---|\n")
for r in all_results:
    if r.get('returncode') == 0:
        grow = str(r.get('grow_step', '-'))
        report.append(
            f"| {r['name']} | {r['seed']} | "
            f"{r.get('final_val_bpb', 'N/A')} | {r.get('final_step', 'N/A')} | "
            f"{r.get('avg_ms_step', 'N/A')} | {grow} |\n"
        )

report_text = ''.join(report)
with open(f'{DRIVE_DIR}/results/progressive_report.md', 'w') as f:
    f.write(report_text)
print(f"Report saved to {DRIVE_DIR}/results/progressive_report.md")
print()
print(report_text)

## Cell 9: Debug — Read log for a specific run

Use this to inspect the full log output if a run failed.

In [ ]:
# Change these to inspect a specific run
DEBUG_NAME = 'smoke_grow_33pct'
DEBUG_SEED = 42
DEBUG_DIR = SMOKE_DIR  # or FULL_DIR for full runs

log_path = f'{DEBUG_DIR}/{DEBUG_NAME}_s{DEBUG_SEED}.txt'
if os.path.exists(log_path):
    with open(log_path) as f:
        lines = f.readlines()
    print(f"Log: {log_path} ({len(lines)} lines)")
    print("=" * 60)

    # Show first 30 lines (startup) and last 30 lines (final results)
    print("--- STARTUP ---")
    for line in lines[:30]:
        print(line.rstrip())

    if len(lines) > 60:
        print(f"\n... ({len(lines) - 60} lines omitted) ...\n")

    print("--- END ---")
    for line in lines[-30:]:
        print(line.rstrip())
else:
    print(f"Log not found: {log_path}")
    print(f"Available logs in {DEBUG_DIR}:")
    for f in sorted(glob.glob(f'{DEBUG_DIR}/*.txt')):
        print(f"  {os.path.basename(f)}")